In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES
import os, sys

import numpy as np
import math

import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

# import xarray as xr
# import uxarray as ux

from tqdm import tqdm

In [ ]:
#Importing DirectoryManager Class

sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "SurfaceVariableAnimations_Unstructured"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class

In [ ]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
###############
#FUNCTIONS

In [ ]:
def GetCLims(ModelData, varNames):
    """
    Compute global (min, max) for one or more variables across all timesteps.
    Returns:
        dict[varName] = (vmin, vmax)
    """
    # Make sure varNames is a list
    if isinstance(varNames, str):
        varNames = [varNames]

    # Initialize dictionary of min/max values
    climDictionary = {v: [np.inf, -np.inf] for v in varNames}

    # Loop over timesteps
    for t in tqdm(range(len(ModelData.fileList)-40), desc="Processing timesteps"):
        data = ModelData.GetDataTimestep(t, printout=False)
        data_diag = ModelData.GetDataTimestep_diag(t, printout=False)

        for varName in varNames:
            variableSubset, _, _ = GetVariable_Subset(varName, data, data_diag)
            vmin = variableSubset.min().item()
            vmax = variableSubset.max().item()

            climDictionary[varName][0] = min(climDictionary[varName][0], vmin)
            climDictionary[varName][1] = max(climDictionary[varName][1], vmax)

        data.close()
        data_diag.close()

    # Convert lists → tuples for readability
    climDictionary = {k: tuple(v) for k, v in climDictionary.items()}
    return climDictionary

In [ ]:
def LatLonBoundingBox_Center(campaign="TRACER"):
    if campaign == "TRACER":
        (latCenter, lonCenter) = 29.67, -95.059
    return latCenter,lonCenter

def LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500):
    # Earth radius in km
    R = 6371.0

    # Convert degrees to radians
    latRadians = math.radians(latCenter)

    # Calculate degree offsets
    dLat = (radius_km / R) * (180.0 / math.pi)
    dLon = (radius_km / (R * math.cos(latRadians))) * (180.0 / math.pi)

    # Bounding box
    latMin = latCenter - dLat
    latMax = latCenter + dLat
    lonMin = lonCenter - dLon
    lonMax = lonCenter + dLon

    latBounds = (latMin, latMax)
    lonBounds = (lonMin, lonMax)
    return latBounds, lonBounds

def LatLonBoundingBox_Subset(variable, latBounds, lonBounds):
    """
    Subset a structured (lat-lon) xarray DataArray or Dataset
    to a given latitude/longitude bounding box.
    """
    # Determine coordinate names (support latitude/lat, longitude/lon)
    lat_name = "latitude" if "latitude" in variable.coords else "lat"
    lon_name = "longitude" if "longitude" in variable.coords else "lon"

    # Handle reversed latitude (if decreasing)
    lat_vals = variable[lat_name].values
    if lat_vals[0] > lat_vals[-1]:
        lat_slice = slice(latBounds[1], latBounds[0])
    else:
        lat_slice = slice(latBounds[0], latBounds[1])

    lon_slice = slice(lonBounds[0], lonBounds[1])

    # Perform subset
    variableSubset = variable.sel({lat_name: lat_slice, lon_name: lon_slice})

    # Extract matching lat/lon arrays
    lat = variableSubset[lat_name].values
    lon = variableSubset[lon_name].values

    # print(f"Subset region: lat={latBounds}, lon={lonBounds}")
    # print(f"Subset shape: {variableSubset[lat_name].shape} × {variableSubset[lon_name].shape}")

    return variableSubset, lat, lon

In [ ]:
def GetVariable(varName,data,data_diag):
    if varName in ModelData.unitsDictionary:
        return data[varName]
    elif varName in ModelData.unitsDictionary_diag:
        return data_diag[varName]
    elif varName == "greenfrac":
        return ModelData.staticData["greenfrac"].isel(nMonths=6)

def GetVariable_Subset(varName,data,data_diag):  
    variable = GetVariable(varName,data,data_diag)
    [latCenter,lonCenter] = LatLonBoundingBox_Center(campaign="TRACER")
    [latBounds, lonBounds] = LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=1000)
    variableSubset, lat, lon = LatLonBoundingBox_Subset(variable,latBounds, lonBounds)

    # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
    return variableSubset, lat, lon

In [ ]:
#PlotVariable_with_Borders()

# Preload map features once
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def PlotVariable_with_Borders(variable, varName, lat,lon, 
                              outputFile=None, save=False, 
                              cmap="viridis", clim=(None,None), 
                              title=None, units=None):
    """
    Plot a uxarray or xarray variable on a map with coastlines, borders, and states,
    using Matplotlib (static PNG output). Works headlessly — no Selenium needed.
    """

    # Create figure
    fig, ax = plt.subplots(
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(9, 5)
    )

    matrix = variable.data
    # Scatter/contour fill (tricontourf works for unstructured grids)
    im = ax.contourf(
        lon, lat, matrix,
        levels=60,
        cmap=cmap,
        vmin=clim[0],vmax=clim[1],
        transform=ccrs.PlateCarree(),
    )

    # Add map features
    ax.add_feature(COAST, linewidth=1)
    ax.add_feature(BORDERS, linewidth=0.8)
    ax.add_feature(STATES, linewidth=0.5)
    ax.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    ax.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Colorbar
    if units is not None:
        label=varName +fr" (${units}$)"
    else: 
        label=varName
    plt.colorbar(im, ax=ax, orientation="vertical", label=label)

    #LABELS
    # Set extent to your data range (forces lat/lon ticks)
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
    
    # Add lat/lon ticks with degrees
    ax.set_xticks(np.linspace(lon.min(), lon.max(), 5), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(lat.min(), lat.max(), 5), crs=ccrs.PlateCarree())
    
    # # Format tick labels as degrees
    # lon_formatter = ccrs.LongitudeFormatter()
    # lat_formatter = ccrs.LatitudeFormatter()
    # ax.xaxis.set_major_formatter(lon_formatter)
    # ax.yaxis.set_major_formatter(lat_formatter)
    if title is not None:
        ax.set_title(title)
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")


    # Save or display
    if save:
        plt.savefig(outputFile, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved static PNG: {outputFile}")
        return None
    else:
        return fig



def SplitTimeString(timeString):
    date, time = timeString.split('_')
    time = time.replace('.', ':')
    return date,time

def GetUnits_Specific(varName):
    for d in (ModelData.unitsDictionary,
              ModelData.unitsDictionary_diag,
              ModelData.unitsDictionary_static):
        if varName in d:
            return d[varName]
    return None
                    
# #TESTING
# t=100
# data = ModelData.GetDataTimestep(t)
# data_diag = ModelData.GetDataTimestep_diag(t); print(f"\n")
# #defining variable names
# varNames = [
#     "q2"]
# # running
# variableDictionary = BuildVariableDictionary(varNames,data,data_diag,climDictionary)
# MakePlots(variableDictionary, save=False)

In [ ]:
def BuildVariableDictionary(varNames,data,data_diag,climDictionary):
    variableDictionary = {}
    for varName in varNames:
        # print(f"Adding {varName}")

        # Getting Output Directory
        folderName = varName
        timeString = ModelData.timeStrings[t]
        fileName = f"{varName}_{timeString}.png"
        outputFile = DirectoryManager.GetOutputFile(outputDirectory, folderName, fileName)
        
        # Handle addition of two variables
        if '+' in varName:
            var1, var2 = varName.split('+')
            var1 = var1.strip()
            var2 = var2.strip()
            
            subset1, lat1, lon1 = GetVariable_Subset(var1,data,data_diag)
            subset2, lat2, lon2 = GetVariable_Subset(var2,data,data_diag)
    
            # Make sure lat/lon are compatible (e.g., same shape)
            if not (np.array_equal(lat1, lat2) and np.array_equal(lon1, lon2)):
                raise ValueError(f"Lat/lon mismatch for {var1} and {var2}")
    
            variableSubset = subset1 + subset2
            lat, lon = lat1, lon1

            # Combine CLims from both variables
            if (var1 in climDictionary) and (var2 in climDictionary):
                vmin = min(climDictionary[var1][0], climDictionary[var2][0])
                vmax = max(climDictionary[var1][1], climDictionary[var2][1])
                clim = (vmin, vmax)
        else:
            variableSubset, lat, lon = GetVariable_Subset(varName,data,data_diag)
            clim = climDictionary[varName]
    
        # Store in dictionary
        variableDictionary[varName] = {
            "data": variableSubset,
            "lat": lat,
            "lon": lon,
            "outputFile": outputFile,
            "clim": clim
        }
        
    return variableDictionary

def MakePlots(variableDictionary, save=False):
    date, time = SplitTimeString(ModelData.timeStrings[t])
    title = f"{ModelData.region}/{ModelData.case}/{ModelData.mpType} on {date} at {time}"
    
    for varName, contents in variableDictionary.items():
        # print(f"Plotting {varName}")
    
        data = contents["data"]
        lat  = contents["lat"]
        lon  = contents["lon"]
        outputFile = contents["outputFile"]
        units = GetUnits_Specific(varName).replace(" ", r"\ ")
        clim = contents["clim"]
        
        fig = PlotVariable_with_Borders(data, varName, lat, lon, 
                                        outputFile, save=save, 
                                        cmap="viridis", clim=clim,
                                        title=title, units=units)

In [ ]:
#################
#RUNNING

In [ ]:
varNames = ["surface_pressure",
            "u10", "v10", "q2",
            "hfx", "qfx", "lh",
            "rainnc", "rainc",
            "refl10cm_1km",
            "greenfrac"]

climDictionary = GetCLims(ModelData,varNames) #run only once

In [ ]:
#time loop
# num_times = ModelData.NTime
num_times = 100
for t in range(num_times):
    if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
    #loading data
    data = ModelData.GetDataTimestep(t)
    data_diag = ModelData.GetDataTimestep_diag(t); print(f"\n")

    #defining variable names
    varNames = [
        "surface_pressure",
        "u10", "v10", "q2",
        "hfx", "qfx", "lh",
        "rainnc+rainc",
        "refl10cm_1km"
    ] + (["greenfrac"] if t == 0 else [])

    # running
    variableDictionary = BuildVariableDictionary(varNames,data,data_diag,climDictionary)
    MakePlots(variableDictionary, save=True)

    break